# t02_position_joint — submit

방향(범주) × slice 깊이(연속 좌표) 실험입니다. 이미지 내부 patch의 x/y 좌표와는 다릅니다.
S11의 고정 Swin feature, 5장 이웃, 3 folds × 3 seeds, loss, head 파라미터 수를 유지합니다.
T00: 기존 방향 embedding + mm RoPE. T01: 깊이 sin/cos 추가. T02: 방향×깊이 상호작용 추가.
학습 후 생성된 bundle로 Internet OFF 제출 notebook을 실행하면 submission.csv가 생성됩니다.


In [ ]:
EXPERIMENT = {'id': 't02_position_joint', 'suite': 'weak', 'backbone': 'swin_t', 'feature_dim': 768, 'slices': 5, 'crop': 224, 'resize': 232, 'interpolation': 'bicubic', 'aggregation': 'class_mil', 'triplet': True, 'protocols': True, 'weak_labels': True, 'finetune': False, 'ensemble': False, 'backbone_epochs': 1, 'epochs': 20, 'batch_size': 16, 'lr': 0.001, 'weight_decay': 0.01, 'expected_training_studies': 4350, 'labels_sha256': '6f704a7bdb2f894cc49445b19ba7c4378c3f548d3449e00361e10044bee40920', 'train_dependencies': ['s11_cross'], 'pipeline_sha256': 'e9a35d51ae44b2a40f83eeae1e5443ba9185fd2023c8a52fa214127b59efd75d', 'architecture': 'spatial_v1', 'representation': 'single', 'neighbors': 5, 'centers': 5, 'hidden_dim': 128, 'heads': 4, 'sequence': True, 'position': 'rope_mm', 'plane_embedding': True, 'fusion': 'cross', 'mm_unit': 3.0, 'feature_pipeline_sha256': '70e4bc4811b24a92c5d0ece041e4e4d5f921c79f507872e6d3f520ce727da0cc', 'plane_depth_mode': 'joint', 'comparison_parent': 't01_position_sum'}


In [ ]:
"""Shared source embedded into each standalone experiment notebook."""

from pathlib import Path

import hashlib

import json

import math

import numpy as np

import pandas as pd

import torch

from torch import nn

PLANES = ("Sagittal", "Coronal", "Axial")

LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
          "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]

SEEDS = (42, 2026, 3407)

def file_hash(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def validate_bundle(path, expected_config):
    path = Path(path).resolve()
    manifest = json.loads((path / "assets.json").read_text(encoding="utf-8"))
    if manifest["experiment"] != expected_config:
        raise ValueError("Wrong experiment bundle: attach this experiment's completed training version.")
    files = manifest["files"]
    wheels = {p.relative_to(path).as_posix() for p in (path / "wheels").glob("*.whl")}
    declared_wheels = {name for name in files if name.startswith("wheels/") and name.endswith(".whl")}
    if (not {"encoder.ts", "model.pt"}.issubset(files) or wheels != declared_wheels
            or len(wheels) != 2
            or not any(name.startswith("wheels/pydicom-3.0.2-") for name in wheels)
            or not any(name.startswith("wheels/python_gdcm-3.2.6-") for name in wheels)):
        raise ValueError("Bundle must include hashes for both models and exactly the two required wheels.")
    for name, digest in manifest["files"].items():
        target = (path / name).resolve()
        if not target.is_relative_to(path) or not target.is_file() or file_hash(target) != digest:
            raise ValueError(f"Invalid or corrupted bundle file: {name}")
    return manifest

def find_data(input_root=Path("/kaggle/input")):
    for path in (input_root / "rsna-knee-abnormality-detection",
                 input_root / "competitions/rsna-knee-abnormality-detection"):
        if (path / "test.csv").is_file():
            return path
    raise FileNotFoundError("Attach the RSNA Knee competition data.")

def find_bundle(input_root=Path("/kaggle/input")):
    paths = list(input_root.glob("*/run/bundle/assets.json"))
    paths += list(input_root.glob("notebooks/*/*/run/bundle/assets.json"))
    paths += list(input_root.glob("kernels/*/*/run/bundle/assets.json"))
    if len(paths) != 1:
        raise ValueError("Attach exactly one completed experiment training output.")
    return paths[0].parent

def make_transform(config):
    from torchvision.transforms import InterpolationMode
    from torchvision.transforms._presets import ImageClassification
    return ImageClassification(crop_size=config["crop"], resize_size=config["resize"],
                               interpolation=InterpolationMode(config["interpolation"]), antialias=True)

def ordered_slices(folder):
    import pydicom
    paths = sorted(folder.glob("*.dcm"))
    if not paths:
        raise FileNotFoundError(f"No DICOM slices in {folder}")
    headers = [pydicom.dcmread(p, stop_before_pixels=True,
               specific_tags=["ImagePositionPatient", "ImageOrientationPatient", "InstanceNumber"]) for p in paths]
    if all(hasattr(h, "ImagePositionPatient") and hasattr(h, "ImageOrientationPatient") for h in headers):
        orientation = np.asarray(headers[0].ImageOrientationPatient, dtype=float)
        normal = np.cross(orientation[:3], orientation[3:])
        if np.linalg.norm(normal) <= .9:
            raise ValueError("Invalid DICOM orientation.")
        coordinates = np.asarray([np.dot(np.asarray(h.ImagePositionPatient, dtype=float), normal) for h in headers])
    elif all(hasattr(h, "InstanceNumber") for h in headers):
        coordinates = np.asarray([float(h.InstanceNumber) for h in headers])
    else:
        raise ValueError(f"No slice position or InstanceNumber: {folder}")
    if not np.isfinite(coordinates).all():
        raise ValueError("Non-finite slice positions.")
    order = np.argsort(coordinates, kind="stable")
    coordinates = coordinates[order]
    span = np.ptp(coordinates)
    positions = (coordinates - coordinates.min()) / span if span > 0 else np.full(len(paths), .5)
    return [paths[i] for i in order], positions.astype(np.float32)

def image_tensor(path, transform):
    import pydicom
    ds = pydicom.dcmread(path)
    pixels = ds.pixel_array.astype(np.float32)
    if pixels.ndim != 2 or not np.isfinite(pixels).all():
        raise ValueError(f"Invalid MRI slice: {path}")
    pixels = pixels * float(getattr(ds, "RescaleSlope", 1)) + float(getattr(ds, "RescaleIntercept", 0))
    if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
        pixels = -pixels
    low, high = np.percentile(pixels, [1, 99])
    if high <= low:
        low, high = float(pixels.min()), float(pixels.max())
    if high <= low:
        raise ValueError(f"Constant MRI slice: {path}")
    pixels = np.clip((pixels - low) / (high - low), 0, 1)
    return transform(torch.from_numpy(pixels).float().unsqueeze(0).repeat(3, 1, 1))

@torch.inference_mode()
def study_tokens(uid, series, image_root, encoder, transform, config, device):
    count, dim = config["slices"], config["feature_dim"]
    features = torch.zeros(3, count, dim)
    mask = torch.zeros(3, count, dtype=torch.bool)
    positions = torch.zeros(3, count)
    rows = series.loc[series.StudyInstanceUID == uid]
    for plane_index, plane in enumerate(PLANES):
        choices = rows.loc[rows.Anatomical_Plane == plane].sort_values(
            ["Fluid_Sensitive", "Fat_Suppression", "SeriesInstanceUID"], ascending=[False, False, True])
        if choices.empty:
            continue
        paths, coords = ordered_slices(image_root / uid / choices.iloc[0].SeriesInstanceUID)
        indices = np.unique(np.rint(np.linspace(.25, .75, count) * (len(paths) - 1)).astype(int))
        batch = torch.stack([image_tensor(paths[i], transform) for i in indices]).to(device)
        output = encoder(batch).cpu().float()
        if output.shape != (len(indices), dim) or not torch.isfinite(output).all():
            raise ValueError("Unexpected encoder features.")
        features[plane_index, :len(indices)] = output
        mask[plane_index, :len(indices)] = True
        positions[plane_index, :len(indices)] = torch.from_numpy(coords[indices])
    if not mask.any():
        raise ValueError(f"No supported MRI planes for study {uid}")
    return features, mask, positions

def pooled_features(tokens, mask):
    clean = torch.where(mask[..., None], tokens, 0.)
    return (clean.sum(2) / mask.sum(2).clamp_min(1)[..., None]).flatten(1)

class StudyHead(nn.Module):
    def __init__(self, config, mean, std):
        super().__init__()
        self.mode = config["aggregation"]
        self.position = config["position"]
        self.register_buffer("mean", mean.clone())
        self.register_buffer("std", std.clone())
        if self.mode == "mean":
            self.classifier = nn.Linear(3 * config["feature_dim"], 12)
        elif self.mode == "attention":
            self.project = nn.Linear(config["feature_dim"], 64)
            layer = nn.TransformerEncoderLayer(64, 4, 128, dropout=0., batch_first=True, norm_first=True)
            self.attention = nn.TransformerEncoder(layer, 1, enable_nested_tensor=False)
            self.classifier = nn.Linear(3 * 64, 12)
            # Initialize common layers identically before adding optional plane parameters.
            self.plane_embedding = nn.Parameter(torch.zeros(3, 64)) if config["plane_embedding"] else None
            self.register_buffer("frequencies", torch.exp(torch.arange(0, 64, 2) * (-math.log(10000.) / 64)))
        else:
            raise ValueError(f"Unknown aggregation: {self.mode}")

    def forward(self, tokens, mask, positions):
        if self.mode == "mean":
            return self.classifier((pooled_features(tokens, mask) - self.mean) / self.std)
        clean = torch.where(mask[..., None], tokens, 0.)
        x = self.project((clean - self.mean) / self.std)
        if self.position:
            angles = positions[..., None] * (2 * math.pi) * self.frequencies
            x = x + torch.stack((angles.sin(), angles.cos()), dim=-1).flatten(-2)
        if self.plane_embedding is not None:
            x = x + self.plane_embedding[None, :, None, :]
        b, planes, slices, dim = x.shape
        flat_mask = mask.reshape(b * planes, slices)
        safe_mask = flat_mask.clone()
        safe_mask[~safe_mask.any(1), 0] = True
        x = self.attention(x.reshape(b * planes, slices, dim), src_key_padding_mask=~safe_mask)
        x = torch.where(flat_mask[..., None], x, 0.)
        vectors = x.sum(1) / flat_mask.sum(1).clamp_min(1)[:, None]
        return self.classifier(vectors.reshape(b, planes * dim))

def make_folds(ids, y):
    from sklearn.model_selection import KFold
    if len(set(ids)) != len(ids) or len(y) != len(ids) or not np.isin(y, [0, 1]).all():
        raise ValueError("Expected unique IDs and complete binary labels.")
    order = np.argsort(ids)
    folds = np.full(len(ids), -1, dtype=int)
    for fold, (train, valid) in enumerate(KFold(3, shuffle=True, random_state=42).split(order)):
        for index in (order[train], order[valid]):
            count = y[index].sum(0)
            if not ((count > 0) & (count < len(index))).all():
                raise ValueError("Each fold/label needs positives and negatives.")
        folds[order[valid]] = fold
    return folds

def metric_scores(y, logits):
    from sklearn.metrics import roc_auc_score
    probabilities = logits.sigmoid().numpy()
    aucs = roc_auc_score(y, probabilities, average=None)
    return {"macro_auc": float(np.mean(aucs)), "per_label_auc": dict(zip(LABELS, aucs.tolist())),
            "bce": nn.functional.binary_cross_entropy_with_logits(logits, torch.from_numpy(y)).item()}

def save_submission(ids, probabilities, sample_columns, target):
    probabilities = np.asarray(probabilities)
    if (not ids or len(set(ids)) != len(ids) or list(sample_columns) != ["StudyInstanceUID", *LABELS]
            or probabilities.shape != (len(ids), 12) or not np.isfinite(probabilities).all()
            or not ((probabilities >= 0) & (probabilities <= 1)).all()):
        raise ValueError("Invalid submission IDs, label order, shape or probabilities.")
    frame = pd.DataFrame(probabilities, columns=LABELS)
    frame.insert(0, "StudyInstanceUID", ids)
    target = Path(target)
    temporary = target.with_suffix(".csv.part")
    frame.to_csv(temporary, index=False)
    temporary.replace(target)
    return frame


In [ ]:
"""Report-supervised MRI experiments; shared verbatim by train and offline submit."""

import re

def per_label_weighted_bce(logits, targets, weights):
    loss = torch.nn.functional.binary_cross_entropy_with_logits(logits.float(), targets.float(), reduction='none')
    denominator = weights.sum(0)
    valid = denominator > 0
    values = (loss * weights).sum(0) / denominator.clamp_min(1e-8)
    return (values * valid).sum() / valid.sum().clamp_min(1)

def validate_weak_bundle(path, config):
    path = Path(path).resolve()
    manifest = json.loads((path / 'assets.json').read_text())
    if manifest['experiment'] != config or manifest['labels'] != LABELS:
        raise ValueError('Wrong weak experiment bundle.')
    required = ({f'members/{eid}/assets.json' for eid in config['train_dependencies']}
                if config.get('ensemble') else {'encoder.ts', 'model.pt'})
    files = manifest['files']
    wheels = {p.relative_to(path).as_posix() for p in (path / 'wheels').glob('*.whl')}
    if (not required <= files.keys() or len(wheels) != 2 or not wheels <= files.keys()
            or not any(x.startswith('wheels/pydicom-3.0.2-') for x in wheels)
            or not any(x.startswith('wheels/python_gdcm-3.2.6-') for x in wheels)):
        raise ValueError('Missing weights or offline DICOM dependencies.')
    for name, digest in files.items():
        target = (path / name).resolve()
        if not target.is_relative_to(path) or not target.is_file() or file_hash(target) != digest:
            raise ValueError(f'Invalid bundle artifact: {name}')
    if config.get('ensemble'):
        for eid in config['train_dependencies']:
            child = (path / 'members' / eid).resolve()
            if not child.is_relative_to(path): raise ValueError('Invalid member path.')
            child_config = json.loads((child / 'assets.json').read_text())['experiment']
            if child_config['id'] != eid: raise ValueError('Wrong ensemble member.')
            validate_weak_bundle(child, child_config)
    return manifest


In [ ]:
"""Slice tokens with explicit coordinates; identical training and inference path."""

def spatial_order(folder):
    import pydicom
    paths = sorted(folder.glob('*.dcm'))
    if not paths:
        raise FileNotFoundError(folder)
    headers = [pydicom.dcmread(p, stop_before_pixels=True, specific_tags=[
        'ImagePositionPatient', 'ImageOrientationPatient', 'InstanceNumber']) for p in paths]
    physical = all(hasattr(h, 'ImagePositionPatient') and hasattr(h, 'ImageOrientationPatient') for h in headers)
    if physical:
        orientation = np.asarray(headers[0].ImageOrientationPatient, dtype=float)
        if any(not np.allclose(h.ImageOrientationPatient, orientation, atol=1e-4) for h in headers):
            raise ValueError('Inconsistent orientations inside one series.')
        normal = np.cross(orientation[:3], orientation[3:])
        if not .99 < np.linalg.norm(normal) < 1.01:
            raise ValueError('Invalid DICOM orientation.')
        values = np.array([np.dot(h.ImagePositionPatient, normal) for h in headers])
    elif all(hasattr(h, 'InstanceNumber') for h in headers):
        values = np.array([float(h.InstanceNumber) for h in headers])
    else:
        raise ValueError('Missing physical and instance coordinates.')
    if not np.isfinite(values).all():
        raise ValueError('Non-finite DICOM coordinates.')
    order = np.argsort(values, kind='stable')
    coords = np.zeros((len(paths), 3), dtype=np.float32)
    coords[:, 0] = np.arange(len(paths))
    coords[:, 1] = values[order] - values[order][0] if physical else 0.
    coords[:, 2] = float(physical)
    return [paths[i] for i in order], coords

def selected_windows(length, config):
    if length < 1 or config['neighbors'] not in (3, 5, 10) or config['centers'] < 1:
        raise ValueError('Invalid slice selection.')
    start, end = config.get('center_range', [.25, .75])
    if not 0 <= start < end <= 1: raise ValueError('Invalid center range.')
    centers = np.unique(np.rint(np.linspace(start, end, config['centers']) * (length-1)).astype(int))
    if config['representation'] == 'triplet':
        if config['neighbors'] != 3:
            raise ValueError('Triplet encoder requires three channels.')
        return [tuple(np.clip([c-1,c,c+1], 0, length-1)) for c in centers]
    if config['representation'] != 'single':
        raise ValueError('Unknown representation.')
    left = (config['neighbors'] - 1) // 2
    indices = {int(np.clip(c+j, 0, length-1)) for c in centers
               for j in range(-left, config['neighbors']-left)}
    return [(i,) for i in sorted(indices)]

def spatial_views(uid, series, image_root, transform, config, audit=None):
    count = config['centers'] * (config['neighbors'] if config['representation'] == 'single' else 1)
    images = torch.zeros(6, count, 3, config['crop'], config['crop'])
    mask = torch.zeros(6, count, dtype=torch.bool)
    coords = torch.zeros(6, count, 3)
    rows = series.loc[series.StudyInstanceUID == uid]
    for slot, (plane, fs) in enumerate((p, f) for p in PLANES for f in (True, False)):
        choices = rows.loc[(rows.Anatomical_Plane == plane) & (rows.Fat_Suppression == fs)]
        choices = choices.sort_values(['Fluid_Sensitive', 'SeriesInstanceUID'], ascending=[False, True])
        if choices.empty:
            continue
        paths, positions = spatial_order(image_root / uid / choices.iloc[0].SeriesInstanceUID)
        decoded = {}
        windows=selected_windows(len(paths),config)
        if audit is not None:
            indices=sorted({i for window in windows for i in window})
            audit.append(dict(StudyInstanceUID=uid,slot=slot,plane=plane,fat_suppression=fs,
                original_slices=len(paths),unique_selected_slices=len(indices),tokens=len(windows),
                selected_index_min=min(indices),selected_index_max=max(indices),valid_fraction=len(windows)/count))
        for j, window in enumerate(windows):
            for index in window:
                if index not in decoded:
                    decoded[index] = image_tensor(paths[index], lambda x:x)[0]
            image = torch.stack([decoded[i] for i in window]) if len(window) == 3 else decoded[window[0]][None].repeat(3,1,1)
            images[slot, j] = transform(image)
            coords[slot, j] = torch.from_numpy(positions[window[len(window)//2]])
            mask[slot, j] = True
    if not mask.any():
        raise ValueError(f'No supported images for {uid}')
    return images, mask, coords

def spatial_signature(config):
    signature = {k: config[k] for k in ('representation', 'neighbors', 'centers', 'crop',
                                       'resize', 'interpolation', 'pipeline_sha256')}
    signature['pipeline_sha256'] = config.get('feature_pipeline_sha256', config['pipeline_sha256'])
    # Legacy configs used this exact range; keep their serialized cache signature valid.
    if list(config.get('center_range', [.25,.75])) != [.25,.75]:
        signature['center_range'] = list(config['center_range'])
    return signature

@torch.inference_mode()
def spatial_tokens(uid, series, image_root, encoder, transform, config, device, audit=None):
    images, mask, coords = spatial_views(uid, series, image_root, transform, config, audit=audit)
    output = torch.cat([encoder(x.to(device)).cpu().float() for x in images[mask].split(16)])
    if output.shape != (int(mask.sum()), config['feature_dim']) or not torch.isfinite(output).all():
        raise ValueError('Invalid spatial features.')
    tokens = torch.zeros(*mask.shape, config['feature_dim'], dtype=torch.float16)
    tokens[mask] = output.half()
    return tokens, mask, coords

def rotary(x, position):
    dim = x.shape[-1]
    if dim % 2:
        raise ValueError('Rotary head dimension must be even.')
    frequencies = torch.exp(torch.arange(0, dim, 2, device=x.device, dtype=torch.float32) * (-math.log(10000.) / dim))
    angle = position.float()[:,None,:,None] * frequencies
    a, b = x.float()[...,0::2], x.float()[...,1::2]
    return torch.stack((a*angle.cos()-b*angle.sin(), a*angle.sin()+b*angle.cos()), dim=-1).flatten(-2).to(x.dtype)

class SliceBlock(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.heads = heads
        self.norm1, self.norm2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.qkv, self.out = nn.Linear(dim, 3*dim), nn.Linear(dim, dim)
        self.ffn = nn.Sequential(nn.Linear(dim,2*dim),nn.GELU(),nn.Linear(2*dim,dim))

    def forward(self, x, mask, position, use_rope):
        b, t, d = x.shape
        q,k,v = self.qkv(self.norm1(x)).reshape(b,t,3,self.heads,d//self.heads).permute(2,0,3,1,4).unbind(0)
        if use_rope:
            q,k = rotary(q,position),rotary(k,position)
        scores = (q.float() @ k.float().transpose(-1,-2)) / math.sqrt(d//self.heads)
        scores = scores.masked_fill(~mask[:,None,None,:], -1e4).softmax(-1)
        context = (scores.to(v.dtype) @ v).transpose(1,2).reshape(b,t,d)
        x = x + self.out(context)
        return (x + self.ffn(self.norm2(x))) * mask[...,None]

class SpatialHead(nn.Module):
    def __init__(self, config, mean, std):
        super().__init__()
        self.config = config
        d = config['hidden_dim']
        if d % config['heads'] or (d//config['heads']) % 2:
            raise ValueError('Invalid attention dimensions.')
        self.register_buffer('mean', mean.float().clone())
        self.register_buffer('std', std.float().clone())
        self.project = nn.Linear(config['feature_dim'], d)
        self.pool_score = nn.Sequential(nn.Linear(d,64),nn.Tanh(),nn.Linear(64,12))
        self.class_weight = nn.Parameter(torch.empty(12,6*d))
        self.class_bias = nn.Parameter(torch.zeros(12))
        nn.init.normal_(self.class_weight,std=.01)
        self.coord_kind = nn.Embedding(2,d)
        nn.init.zeros_(self.coord_kind.weight)
        if config['sequence']:
            self.slice_block = SliceBlock(d,config['heads'])
        if config['plane_embedding']:
            self.plane = nn.Embedding(3,d)
            self.protocol = nn.Embedding(2,d)
        mode = config['fusion']
        if mode == 'mlp':
            self.mlp = nn.Sequential(nn.Linear(6*d,d),nn.GELU(),nn.Linear(d,1))
        elif mode == 'gate':
            self.gate = nn.Linear(d,1)
            self.gate_weight = nn.Parameter(torch.randn(12,d)*.01)
        elif mode == 'cross':
            self.queries = nn.Parameter(torch.randn(12,d)*.02)
            self.memory_norm, self.query_norm = nn.LayerNorm(d),nn.LayerNorm(d)
            self.cross = nn.MultiheadAttention(d,config['heads'],batch_first=True,dropout=0.)
            self.read_norm = nn.LayerNorm(d)
            self.read_ffn = nn.Sequential(nn.Linear(d,2*d),nn.GELU(),nn.Linear(2*d,d))
            self.read_weight = nn.Parameter(torch.randn(12,d)*.01)
        elif mode != 'concat':
            raise ValueError('Unsupported fusion.')

    def forward(self, tokens, mask, coords):
        if tokens.ndim != 4 or tokens.shape[1] != 6 or mask.shape != tokens.shape[:3] or coords.shape != (*mask.shape,3):
            raise ValueError('Invalid token/coordinate shape.')
        if not mask.flatten(1).any(1).all():
            raise ValueError('Each study requires a valid image.')
        coords = torch.where(mask[...,None],coords,0.)
        clean = torch.where(mask[...,None],tokens.float(),self.mean)
        if not torch.isfinite(coords).all() or not torch.isfinite(clean).all():
            raise ValueError('Non-finite valid token or coordinate.')
        x = self.project((clean-self.mean)/self.std)
        x = x + self.coord_kind((coords[...,2]>.5).long())
        b,s,t,d = x.shape
        if self.config['plane_embedding']:
            slot=torch.arange(6,device=x.device)
            x=x+(self.plane(slot//2)+self.protocol(slot%2))[None,:,None,:]
        pos=coords[...,0]
        if self.config['position']=='rope_mm':
            pos=torch.where(coords[...,2]>.5,coords[...,1]/self.config['mm_unit'],pos)
        if self.config['position']=='additive':
            freq=torch.exp(torch.arange(0,d,2,device=x.device)*(-math.log(10000.)/d))
            angles=pos[...,None]*freq
            x=x+torch.stack((angles.sin(),angles.cos()),-1).flatten(-2)
        x=x*mask[...,None]
        if self.config['sequence']:
            flat=x.reshape(b*s,t,d); flat_mask=mask.reshape(b*s,t); active=flat_mask.any(1)
            result=torch.zeros_like(flat)
            result[active]=self.slice_block(flat[active],flat_mask[active],pos.reshape(b*s,t)[active],self.config['position'].startswith('rope'))
            x=result.reshape(b,s,t,d)
        if self.config['fusion']=='cross':
            memory=self.memory_norm(x.flatten(1,2)); query=self.queries[None].expand(b,-1,-1)
            read,_=self.cross(self.query_norm(query),memory,memory,key_padding_mask=~mask.flatten(1),need_weights=False)
            read=query+read; read=read+self.read_ffn(self.read_norm(read))
            return (read*self.read_weight).sum(-1)+self.class_bias
        attention=self.pool_score(x).masked_fill(~mask[...,None],-1e4).softmax(2)*mask[...,None]
        pooled=torch.einsum('bstc,bstd->bscd',attention,x).permute(0,2,1,3)
        if self.config['fusion']=='mlp':
            return self.mlp(pooled.flatten(2)).squeeze(-1)+self.class_bias
        if self.config['fusion']=='gate':
            weight=self.gate(pooled).squeeze(-1).masked_fill(~mask.any(2)[:,None,:],-1e4).softmax(-1)
            return ((pooled*weight[...,None]).sum(2)*self.gate_weight).sum(-1)+self.class_bias
        return (pooled.flatten(2)*self.class_weight).sum(-1)+self.class_bias


In [ ]:
"""Two-factor position ablation: categorical plane x continuous slice depth.

Kept separate so an already running spatial/follow-up queue keeps its source hash.
This head retains S11's per-series RoPE and class-query cross attention.
"""

class PlaneDepthHead(SpatialHead):
    def __init__(self, config, mean, std):
        if (config['fusion'] != 'cross' or not config['sequence'] or
                not config['plane_embedding'] or config['position'] != 'rope_mm'):
            raise ValueError('Plane-depth ablation requires the fixed S11 architecture.')
        if config.get('plane_depth_mode') not in ('none', 'sum', 'joint'):
            raise ValueError('Unknown plane-depth encoding.')
        if not math.isfinite(config['mm_unit']) or config['mm_unit'] <= 0:
            raise ValueError('mm_unit must be positive and finite.')
        super().__init__(config, mean, std)

    def position_features(self, position):
        """(B, 6, T) -> (B, 6, T, D), with no new trainable parameters.

        방향은 0/1/2의 연속 좌표가 아닌 Sagittal/Coronal/Axial 범주이다.
        slice 위치는 series 첫 장에서의 mm 거리 / mm_unit; 없으면 정렬 index.
        sum: 방향 embedding과 깊이 sin/cos를 별도로 더한다.
        joint: 방향 embedding으로 깊이 신호를 조절하는 상호작용도 더한다.
        tanh는 조절량을 [-1, 1]로 제한한다. 실제 학습에서는 기존 plane
        embedding으로 gradient가 흐른다. 이것은 2D RoPE나 patch (x,y)가 아니다.
        """
        d = self.config['hidden_dim']
        frequencies = torch.exp(torch.arange(0, d, 2, device=position.device,
                                            dtype=torch.float32) * (-math.log(10000.) / d))
        angles = position.float()[..., None] * frequencies
        depth = torch.stack((angles.sin(), angles.cos()), -1).flatten(-2)
        mode = self.config['plane_depth_mode']
        if mode == 'none':
            return torch.zeros_like(depth)
        if mode == 'sum':
            return depth
        plane = self.plane(torch.arange(6, device=position.device) // 2)
        return depth + torch.tanh(plane)[None, :, None, :] * depth

    def forward(self, tokens, mask, coords):
        # S11 cross branch reproduced here to leave active historical code untouched.
        # A regression test requires exact S11 outputs/gradients in the 'none' control.
        if tokens.ndim != 4 or tokens.shape[1] != 6 or mask.shape != tokens.shape[:3] or coords.shape != (*mask.shape, 3):
            raise ValueError('Invalid token/coordinate shape.')
        if not mask.flatten(1).any(1).all():
            raise ValueError('Each study requires a valid image.')
        coords = torch.where(mask[..., None], coords, 0.)
        clean = torch.where(mask[..., None], tokens.float(), self.mean)
        if not torch.isfinite(coords).all() or not torch.isfinite(clean).all():
            raise ValueError('Non-finite valid token or coordinate.')
        x = self.project((clean - self.mean) / self.std)
        x = x + self.coord_kind((coords[..., 2] > .5).long())
        b, s, t, d = x.shape
        slot = torch.arange(6, device=x.device)
        x = x + (self.plane(slot // 2) + self.protocol(slot % 2))[None, :, None, :]
        pos = torch.where(coords[..., 2] > .5, coords[..., 1] / self.config['mm_unit'], coords[..., 0])
        # NEW: absolute depth is visible to the cross-view class queries as well.
        # Existing RoPE below still supplies relative depth inside each series.
        if self.config['plane_depth_mode'] != 'none':
            x = x + self.position_features(pos)
        x = x * mask[..., None]
        flat = x.reshape(b * s, t, d)
        flat_mask = mask.reshape(b * s, t)
        active = flat_mask.any(1)
        result = torch.zeros_like(flat)
        result[active] = self.slice_block(flat[active], flat_mask[active],
                                         pos.reshape(b * s, t)[active], True)
        x = result.reshape(b, s, t, d)
        memory = self.memory_norm(x.flatten(1, 2))
        query = self.queries[None].expand(b, -1, -1)
        read, _ = self.cross(self.query_norm(query), memory, memory,
                             key_padding_mask=~mask.flatten(1), need_weights=False)
        read = query + read
        read = read + self.read_ffn(self.read_norm(read))
        return (read * self.read_weight).sum(-1) + self.class_bias

SpatialHead = PlaneDepthHead


In [ ]:
"""Offline prediction using the exact trained token/coordinate pipeline."""

import subprocess

import sys

import time

def load_spatial_predictor(bundle,config,device):
    manifest=validate_weak_bundle(bundle,config)
    if manifest['python']!=list(sys.version_info[:2]) or manifest['torch']!=torch.__version__:
        raise ValueError('Use the pinned training runtime.')
    saved=torch.load(bundle/'model.pt',map_location='cpu',weights_only=True)
    if saved['experiment']!=config or saved['labels']!=LABELS: raise ValueError('Wrong trained head.')
    head=SpatialHead(config,saved['state']['mean'],saved['state']['std'])
    head.load_state_dict(saved['state'],strict=True)
    encoder=torch.jit.load(str(bundle/'encoder.ts'),map_location=device).eval()
    return encoder,head.to(device).eval(),make_transform(config),config

def submit_spatial_experiment(config,data,bundle,out):
    from tqdm.auto import tqdm
    data,bundle,out=Path(data),Path(bundle),Path(out); started=time.monotonic()
    validate_weak_bundle(bundle,config)
    subprocess.check_call([sys.executable,'-m','pip','install','-q','--no-index','--no-deps','--find-links',str(bundle/'wheels'),
                           'pydicom==3.0.2','python-gdcm==3.2.6'])
    torch.set_num_threads(1); torch.backends.mha.set_fastpath_enabled(False)
    device='cuda' if torch.cuda.is_available() else 'cpu'; members=[]
    for eid in config['train_dependencies'] if config.get('ensemble') else [None]:
        path=bundle/'members'/eid if eid else bundle
        cfg=json.loads((path/'assets.json').read_text())['experiment']
        members.append(load_spatial_predictor(path,cfg,device))
    test=pd.read_csv(data/'test.csv'); series=pd.read_csv(data/'test_series.csv')
    sample=pd.read_csv(data/'sample_submission.csv',nrows=0)
    if not len(test) or test.StudyInstanceUID.isna().any() or not test.StudyInstanceUID.is_unique or not series.SeriesInstanceUID.is_unique:
        raise ValueError('Invalid test metadata.')
    predictions=[]
    with torch.inference_mode():
        for uid in tqdm(test.StudyInstanceUID,desc=config['id']+' test'):
            probs=[]
            for encoder,head,transform,cfg in members:
                tokens,mask,coords=spatial_tokens(uid,series,data/'test_series',encoder,transform,cfg,device)
                probs.append(head(tokens[None].to(device),mask[None].to(device),coords[None].to(device)).sigmoid().cpu().numpy()[0])
            predictions.append(np.mean(probs,axis=0))
    out.mkdir(parents=True,exist_ok=True)
    frame=save_submission(test.StudyInstanceUID.tolist(),predictions,sample.columns,out/'submission.csv')
    (out/'inference_summary.json').write_text(json.dumps(dict(experiment=config,studies=len(test),labels=12,
        seconds=time.monotonic()-started,manifest_sha256=file_hash(bundle/'assets.json'),internet_required=False),indent=2),encoding='utf-8')
    return frame


In [ ]:
submit_spatial_experiment(EXPERIMENT,find_data(),find_bundle(),Path('/kaggle/working'))
